# Notebook 4: Multi-Asset Research Pipeline

**Alpha Search — Real Data Only**

This notebook combines **ALL free data sources** into a unified research pipeline with robust error handling. It fetches whatever data is available and runs analysis on it — never generating synthetic data.

**Data Sources Used:**
| Source | API | Data | Reliability |
|--------|-----|------|-------------|
| **FRED** | api.stlouisfed.org | GDP, CPI, rates | High — always works |
| **CoinGecko** | api.coingecko.com | BTC, ETH, SOL | High — generous limits |
| **SEC EDGAR** | data.sec.gov | Company financials | Medium — may rate-limit |
| **yfinance** | yfinance lib | Stock OHLCV | Low from Colab — shared IP |

**Error Handling:**
- Each data source has independent try/except blocks
- If yfinance fails, shows clear message about rate limits
- Analysis runs on whatever data was successfully fetched
- No synthetic data is ever generated

In [ ]:
# Install Alpha Search
!pip install git+https://github.com/alpha-search/alpha-search.git -q

In [ ]:
# All Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests
import time
import warnings
import sys
import json
from datetime import datetime
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100
print('All imports loaded successfully.')

In [ ]:
# Robust Data Fetchers — One for each source
# All use free public APIs with no API keys

SEC_HEADERS = {'User-Agent': 'AlphaSearch Research (research@alphasearch.io)'}

# --- FRED (no API key needed) ---
def fetch_fred(series_id, label=None):
    """Fetch FRED data via public CSV endpoint (no API key required)."""
    url = f'https://fred.stlouisfed.org/graph/fredgraph.csv?id={series_id}'
    try:
        df = pd.read_csv(url, parse_dates=['observation_date'])
        df['value'] = pd.to_numeric(df.iloc[:, 1], errors='coerce')
        df = df.dropna(subset=['value'])
        series = df.set_index('observation_date')['value']
        series.name = label or series_id
        print(f"  [OK] FRED {series_id}: {len(series)} observations")
        return series
    except Exception as e:
        print(f"  [FAIL] FRED {series_id}: {e}")
        return pd.Series(name=label or series_id)

# --- CoinGecko (no API key needed) ---
def fetch_crypto(coin_id='bitcoin', days=365, label=None):
    """Fetch crypto price history from CoinGecko (free, no key)."""
    url = f'https://api.coingecko.com/api/v3/coins/{coin_id}/market_chart'
    params = {'vs_currency': 'usd', 'days': days}
    try:
        resp = requests.get(url, params=params, timeout=30)
        resp.raise_for_status()
        data = resp.json()
        if 'prices' not in data:
            print(f"  [FAIL] No price data for {coin_id}")
            return pd.Series(name=label or coin_id)
        prices = pd.DataFrame(data['prices'], columns=['ts', 'price'])
        prices['ts'] = pd.to_datetime(prices['ts'], unit='ms')
        series = prices.set_index('ts')['price']
        series.name = label or coin_id
        print(f"  [OK] CoinGecko {coin_id}: {len(series)} prices")
        return series
    except requests.exceptions.HTTPError as e:
        if resp.status_code == 429:
            print(f"  [RATE LIMIT] CoinGecko {coin_id}. Waiting 60s then retrying...")
            time.sleep(60)
            return fetch_crypto(coin_id, days, label)
        print(f"  [FAIL] CoinGecko {coin_id} HTTP {resp.status_code}: {e}")
        return pd.Series(name=label or coin_id)
    except Exception as e:
        print(f"  [FAIL] CoinGecko {coin_id}: {e}")
        return pd.Series(name=label or coin_id)

# --- SEC EDGAR (no API key needed, requires User-Agent) ---
def get_cik(ticker):
    """Look up CIK for a ticker symbol."""
    url = 'https://www.sec.gov/files/company_tickers.json'
    try:
        resp = requests.get(url, headers=SEC_HEADERS, timeout=30)
        resp.raise_for_status()
        data = resp.json()
        for entry in data.values():
            if entry['ticker'].upper() == ticker.upper():
                return str(entry['cik_str']).zfill(10)
        raise ValueError(f'Ticker {ticker} not found')
    except Exception as e:
        print(f"  [FAIL] CIK lookup for {ticker}: {e}")
        return None

def fetch_sec_facts(cik):
    """Fetch company facts from SEC EDGAR. CIK must be 10-digit zero-padded."""
    url = f'https://data.sec.gov/api/xbrl/companyfacts/CIK{cik}.json'
    try:
        resp = requests.get(url, headers=SEC_HEADERS, timeout=60)
        resp.raise_for_status()
        return resp.json()
    except requests.exceptions.HTTPError as e:
        if resp.status_code == 403:
            print(f"  [FAIL] SEC 403 Forbidden — may be rate-limiting this IP")
        elif resp.status_code == 404:
            print(f"  [FAIL] SEC 404 — No data for CIK {cik}")
        else:
            print(f"  [FAIL] SEC HTTP {resp.status_code}: {e}")
        return None
    except Exception as e:
        print(f"  [FAIL] SEC CIK {cik}: {e}")
        return None

# --- yfinance (may be rate-limited on shared IPs) ---
def fetch_yfinance(ticker='SPY', period='1y', retries=5):
    """Fetch stock data using yfinance with retry logic."""
    try:
        import yfinance as yf
    except ImportError:
        print('  [INFO] Installing yfinance...')
        !pip install yfinance -q
        import yfinance as yf
    for attempt in range(retries):
        try:
            print(f"  [TRY {attempt+1}/{retries}] yfinance: {ticker}")
            stock = yf.Ticker(ticker)
            hist = stock.history(period=period, auto_adjust=True)
            if len(hist) > 0:
                print(f"  [OK] yfinance {ticker}: {len(hist)} rows")
                return hist
            time.sleep(1)
        except Exception as e:
            print(f"  [FAIL] Attempt {attempt+1}: {e}")
            time.sleep(1)
    print(f"  [FAIL] yfinance {ticker}: All {retries} attempts failed.")
    return pd.DataFrame()

print('All fetcher functions defined.')

In [ ]:
# Phase 1: Fetch Macro Data (FRED)
print('=' * 70)
print('PHASE 1: Fetching Macroeconomic Data from FRED')
print('=' * 70)

macro = {}
macro['GDP'] = fetch_fred('GDP', label='GDP')
time.sleep(0.3)
macro['CPI'] = fetch_fred('CPIAUCSL', label='CPI')
time.sleep(0.3)
macro['UNRATE'] = fetch_fred('UNRATE', label='Unemployment')
time.sleep(0.3)
macro['DFF'] = fetch_fred('DFF', label='Fed Funds')
time.sleep(0.3)
macro['T10Y2Y'] = fetch_fred('T10Y2Y', label='Yield Curve')

macro_success = sum(1 for v in macro.values() if len(v) > 0)
print(f'FRED: {macro_success}/5 series fetched successfully.')

In [ ]:
# Phase 2: Fetch Crypto Data (CoinGecko)
print('\n' + '=' * 70)
print('PHASE 2: Fetching Cryptocurrency Data from CoinGecko')
print('=' * 70)

crypto = {}
crypto['BTC'] = fetch_crypto('bitcoin', days=365, label='Bitcoin')
time.sleep(1.5)
crypto['ETH'] = fetch_crypto('ethereum', days=365, label='Ethereum')
time.sleep(1.5)
crypto['SOL'] = fetch_crypto('solana', days=365, label='Solana')

crypto_success = sum(1 for v in crypto.values() if len(v) > 0)
print(f'CoinGecko: {crypto_success}/3 coins fetched successfully.')

In [ ]:
# Phase 3: Fetch Company Fundamentals (SEC EDGAR)
print('\n' + '=' * 70)
print('PHASE 3: Fetching Company Fundamentals from SEC EDGAR')
print('=' * 70)

sec_data = {}

# Fetch AAPL fundamentals as a demonstration
ticker = 'AAPL'
print(f'Fetching SEC data for {ticker}...')

try:
    cik = get_cik(ticker)
    if cik:
        print(f'  CIK for {ticker}: {cik}')
        time.sleep(0.5)
        facts = fetch_sec_facts(cik)
        if facts and 'facts' in facts:
            sec_data[ticker] = facts
            print(f'  [OK] SEC {ticker}: Fetched company facts')
        else:
            print(f'  [FAIL] SEC {ticker}: Could not fetch company facts')
    else:
        print(f'  [FAIL] Could not look up CIK for {ticker}')
except Exception as e:
    print(f'  [FAIL] SEC EDGAR error: {e}')

if not sec_data:
    print()
    print('!' * 70)
    print('SEC EDGAR UNAVAILABLE')
    print('!' * 70)
    print('The SEC API may be rate-limiting this shared IP.')
    print('This is common in cloud environments.')
    print('The notebook will continue with other data sources.')
    print('!' * 70)

sec_success = len(sec_data)
print(f'SEC EDGAR: {sec_success} companies fetched.')

In [ ]:
# Phase 4: Try to Fetch Stock Data (yfinance with retries)
print('\n' + '=' * 70)
print('PHASE 3: Fetching Stock Data from Yahoo Finance')
print('=' * 70)

stocks = {}
stocks['SPY'] = fetch_yfinance('SPY', period='1y', retries=5)
time.sleep(1)
stocks['AAPL'] = fetch_yfinance('AAPL', period='1y', retries=5)

stocks_success = sum(1 for v in stocks.values() if len(v) > 0)
print(f'\nyfinance: {stocks_success}/2 stocks fetched successfully.')

if stocks_success == 0:
    print()
    print('!' * 70)
    print('YAHOO FINANCE RATE-LIMITED')
    print('!' * 70)
    print()
    print('Yahoo Finance is rate-limiting this shared IP (common in Colab).')
    print()
    print('SOLUTIONS:')
    print('  1. Run this notebook locally (not on shared Colab IP)')
    print('  2. Set ALPHA_VANTAGE_API_KEY env var for stock data')
    print('  3. Use FRED and CoinGecko data (which work reliably)')
    print()
    print('The notebook will continue with available data sources.')
    print('!' * 70)

In [ ]:
# Phase 5: Data Summary — What We Have
print('\n' + '=' * 70)
print('DATA AVAILABILITY SUMMARY')
print('=' * 70)

all_sources = {
    'FRED (Macro)': macro,
    'CoinGecko (Crypto)': crypto,
    'yfinance (Stocks)': stocks,
}

available_count = 0
for source_name, source_data in all_sources.items():
    print(f'\n{source_name}:')
    for series_name, series_data in source_data.items():
        if isinstance(series_data, pd.DataFrame):
            status = f'{len(series_data)} rows' if len(series_data) > 0 else 'FAILED'
            if len(series_data) > 0:
                available_count += 1
        else:
            status = f'{len(series_data)} obs, latest={series_data.iloc[-1]:.2f}' if len(series_data) > 0 else 'FAILED'
            if len(series_data) > 0:
                available_count += 1
        print(f'  {series_name}: {status}')

print(f'\nTotal available data series: {available_count}')
print('Analysis will proceed with whatever data was successfully fetched.')

In [ ]:
# Phase 6: Run Analysis Pipeline on Available Data
print('\n' + '=' * 70)
print('ANALYSIS PIPELINE: Running on Available Data')
print('=' * 70)

if macro_success >= 3:
    print('\n[1] Macro Dashboard — Available')
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle('Macro Dashboard (FRED)', fontsize=16, fontweight='bold')
    if len(macro['GDP']) > 0:
        ax = axes[0, 0]
        gdp = macro['GDP']
        ax.fill_between(gdp.index, gdp.values, alpha=0.3, color='#1f77b4')
        ax.plot(gdp.index, gdp.values, color='#1f77b4', linewidth=2)
        ax.set_title('GDP (Billions USD)')
        ax.set_ylabel('Billions $')
        ax.grid(True, alpha=0.3)
    if len(macro['CPI']) > 0:
        ax = axes[0, 1]
        inf = macro['CPI'].pct_change(12) * 100
        ax.fill_between(inf.index, inf.values, 0, alpha=0.3, color='green')
        ax.plot(inf.index, inf.values, color='darkgreen', linewidth=1)
        ax.axhline(y=2, color='orange', linestyle='--')
        ax.set_title('Inflation (CPI YoY %)')
        ax.set_ylabel('Percent')
        ax.grid(True, alpha=0.3)
    if len(macro['UNRATE']) > 0:
        ax = axes[1, 0]
        ax.plot(macro['UNRATE'].index, macro['UNRATE'].values, color='#ff7f0e', linewidth=2)
        ax.set_title('Unemployment Rate (%)')
        ax.set_ylabel('Percent')
        ax.grid(True, alpha=0.3)
    if len(macro['DFF']) > 0:
        ax = axes[1, 1]
        ax.plot(macro['DFF'].index, macro['DFF'].values, color='#d62728', linewidth=1.5)
        ax.fill_between(macro['DFF'].index, macro['DFF'].values, alpha=0.2, color='#d62728')
        ax.set_title('Fed Funds Rate (%)')
        ax.set_ylabel('Percent')
        ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    print('   Macro dashboard plotted.')
else:
    print('\n[1] Macro Dashboard — Skipped (insufficient FRED data)')

if crypto_success >= 2:
    print('\n[2] Crypto Performance — Available')
    fig, ax = plt.subplots(figsize=(12, 6))
    colors = {'BTC': '#F7931A', 'ETH': '#627EEA', 'SOL': '#14F195'}
    for ticker, prices in crypto.items():
        if len(prices) > 0:
            norm = (prices / prices.iloc[0]) * 100
            ax.plot(norm.index, norm.values, label=ticker,
                    color=colors.get(ticker, 'gray'), linewidth=2)
    ax.axhline(y=100, color='black', linestyle='--', alpha=0.5)
    ax.set_title('Crypto Performance (Normalized to 100)', fontsize=14, fontweight='bold')
    ax.set_ylabel('Normalized Price')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    vols = {}
    for ticker, prices in crypto.items():
        if len(prices) > 0:
            ret = prices.resample('D').last().dropna().pct_change().dropna()
            vols[ticker] = ret.std() * np.sqrt(365) * 100
    if vols:
        fig, ax = plt.subplots(figsize=(8, 5))
        bars = ax.bar(vols.keys(), vols.values(),
                      color=[colors.get(k, 'gray') for k in vols.keys()], alpha=0.8)
        ax.set_title('Annualized Volatility (%)', fontsize=14, fontweight='bold')
        ax.set_ylabel('Volatility (%)')
        for bar, val in zip(bars, vols.values()):
            ax.text(bar.get_x() + bar.get_width()/2., val, f'{val:.0f}%',
                    ha='center', va='bottom', fontweight='bold')
        plt.tight_layout()
        plt.show()
    print('   Crypto analysis complete.')
else:
    print('\n[2] Crypto Performance — Skipped (insufficient CoinGecko data)')

In [ ]:
# Phase 7: Stock Analysis (if yfinance worked)
if stocks_success > 0:
    print('\n[3] Stock Analysis — Available')
    for ticker, df in stocks.items():
        if len(df) == 0:
            continue
        df['SMA20'] = df['Close'].rolling(20).mean()
        df['SMA50'] = df['Close'].rolling(50).mean()
        df['Signal'] = np.where(df['SMA20'] > df['SMA50'], 'BULLISH', 'BEARISH')
        fig, ax = plt.subplots(figsize=(12, 6))
        ax.plot(df.index, df['Close'], label='Close', linewidth=1.5, color='black')
        ax.plot(df.index, df['SMA20'], label='SMA 20', linewidth=1, color='blue', alpha=0.7)
        ax.plot(df.index, df['SMA50'], label='SMA 50', linewidth=1, color='red', alpha=0.7)
        latest_signal = df['Signal'].iloc[-1]
        ax.set_title(f'{ticker} Signal: {latest_signal}', fontsize=14, fontweight='bold')
        ax.set_ylabel('Price (USD)')
        ax.legend()
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()
        print(f'   {ticker}: {latest_signal} | Close=${df["Close"].iloc[-1]:.2f}')
else:
    print('\n[3] Stock Analysis — Skipped (yfinance rate-limited or unavailable)')

In [ ]:
# Phase 8: Cross-Asset Summary Dashboard
print('\n' + '=' * 70)
print('CROSS-ASSET SUMMARY DASHBOARD')
print('=' * 70)

fig = plt.figure(figsize=(16, 10))
gs = fig.add_gridspec(3, 3, hspace=0.3, wspace=0.3)

if len(macro.get('UNRATE', [])) > 0:
    ax = fig.add_subplot(gs[0, 0])
    ax.plot(macro['UNRATE'].index, macro['UNRATE'].values, color='#ff7f0e', linewidth=1.5)
    ax.set_title('Unemployment (%)', fontweight='bold', fontsize=10)
    ax.grid(True, alpha=0.3)

if len(macro.get('DFF', [])) > 0:
    ax = fig.add_subplot(gs[0, 1])
    ax.plot(macro['DFF'].index, macro['DFF'].values, color='#d62728', linewidth=1.5)
    ax.set_title('Fed Funds Rate (%)', fontweight='bold', fontsize=10)
    ax.grid(True, alpha=0.3)

if len(macro.get('T10Y2Y', [])) > 0:
    ax = fig.add_subplot(gs[0, 2])
    yc = macro['T10Y2Y']
    ax.plot(yc.index, yc.values, color='#9467bd', linewidth=1.5)
    ax.fill_between(yc.index, yc.values, 0, where=(yc.values < 0), alpha=0.4, color='red')
    ax.axhline(y=0, color='black', linewidth=0.5)
    ax.set_title('Yield Curve (10Y-2Y)', fontweight='bold', fontsize=10)
    ax.grid(True, alpha=0.3)

ax_crypto = fig.add_subplot(gs[1, :])
colors = {'BTC': '#F7931A', 'ETH': '#627EEA', 'SOL': '#14F195'}
for ticker, prices in crypto.items():
    if len(prices) > 0:
        norm = (prices / prices.iloc[0]) * 100
        ax_crypto.plot(norm.index, norm.values, label=ticker,
                       color=colors.get(ticker, 'gray'), linewidth=2)
ax_crypto.set_title('Crypto Performance (Normalized)', fontweight='bold', fontsize=12)
ax_crypto.legend()
ax_crypto.grid(True, alpha=0.3)

ax_text = fig.add_subplot(gs[2, :])
ax_text.axis('off')
lines = ['DATA SOURCES STATUS:', '=' * 50]
lines.append(f'  FRED Macro:    {macro_success}/5 series OK')
lines.append(f'  CoinGecko:     {crypto_success}/3 coins OK')
lines.append(f'  Yahoo Finance: {stocks_success}/2 stocks OK')
lines.append('')
if len(macro.get('UNRATE', [])) > 0:
    lines.append(f'UNEMPLOYMENT: {macro["UNRATE"].iloc[-1]:.1f}%')
if len(macro.get('DFF', [])) > 0:
    lines.append(f'FED FUNDS:    {macro["DFF"].iloc[-1]:.2f}%')
if len(macro.get('T10Y2Y', [])) > 0:
    yc_val = macro['T10Y2Y'].iloc[-1]
    lines.append(f'YIELD CURVE:  {yc_val:.2f}pp ({"INVERTED" if yc_val < 0 else "Normal"})')
lines.append('')
for ticker, prices in crypto.items():
    if len(prices) > 0:
        ret = (prices.iloc[-1] / prices.iloc[0] - 1) * 100
        lines.append(f'{ticker}: ${prices.iloc[-1]:,.0f} ({ret:+.1f}%)')
summary_text = '\n'.join(lines)
ax_text.text(0.1, 0.5, summary_text, transform=ax_text.transAxes, fontsize=10,
             verticalalignment='center', fontfamily='monospace',
             bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.3))
fig.suptitle('Multi-Asset Research Pipeline — All Real Data', fontsize=16, fontweight='bold', y=0.98)
plt.show()
print('Dashboard complete.')

In [ ]:
# Phase 9: Save Results
print('\n' + '=' * 70)
print('SAVING RESULTS')
print('=' * 70)
results = {
    'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'macro_summary': {},
    'crypto_summary': {},
}
for name, series in macro.items():
    if len(series) > 0:
        results['macro_summary'][name] = {
            'latest': float(series.iloc[-1]),
            'observations': len(series),
            'period': f'{series.index.min().date()} to {series.index.max().date()}'
        }
for name, series in crypto.items():
    if len(series) > 0:
        ret = (series.iloc[-1] / series.iloc[0] - 1) * 100
        daily_ret = series.resample('D').last().dropna().pct_change().dropna()
        vol = daily_ret.std() * np.sqrt(365) * 100 if len(daily_ret) > 0 else 0
        results['crypto_summary'][name] = {
            'latest_price': float(series.iloc[-1]),
            'period_return_pct': float(ret),
            'annualized_vol_pct': float(vol),
        }
results_json = json.dumps(results, indent=2, default=str)
print('Results summary:')
print(results_json[:2000])
print('...')
print(f'\nResults can be saved to file or passed to AlphaSearch MemoryStore.')

In [ ]:
# Final Summary
print('=' * 70)
print('MULTI-ASSET PIPELINE: EXECUTION COMPLETE')
print('=' * 70)
print()
print(f'Data Sources Attempted: 4')
print(f'  FRED (Macro):     {macro_success}/5 series OK')
print(f'  CoinGecko:        {crypto_success}/3 coins OK')
print(f'  Yahoo Finance:    {stocks_success}/2 stocks OK')
print()
print('Analysis Performed:')
print('  1. Macro dashboard (GDP, inflation, unemployment, rates)')
print('  2. Crypto performance and volatility comparison')
print('  3. Stock SMA crossover signals (if available)')
print('  4. Cross-asset summary dashboard')
print('  5. Results aggregation and export')
print()
print('Rate Limit Notes:')
print('  - FRED: ~120 req/min (very generous)')
print('  - CoinGecko: 10-30 req/min (use 1.5s delays)')
print('  - Yahoo Finance: varies (shared IP may be limited)')
print('  - SEC EDGAR: ~10 req/sec (requires User-Agent)')
print()
print('NO SYNTHETIC DATA WAS USED IN THIS NOTEBOOK.')
print('All numbers come from live API calls.')
print('If an API failed, the error was reported, not hidden.')
print()
print('=' * 70)